In [ ]:
import oncophylo as op
import numpy as np
import pandas as pd
import anndata as ad
import os, sys
import matplotlib.pyplot as plt
from pathlib import Path
from anndata import io
import h5py

In [ ]:
import json 
import networkx as nx 

def load_tree(adata, name):
    return nx.node_link_graph(json.loads(adata.uns[name]))

In [ ]:
import re
from collections import defaultdict, deque

def resolve_mutation_relationships(T, _type="simulation"):
    """
    Extract SNVs/CNAs from a mutation tree T, and compute:
      1. Ancestor–descendant mutation pairs
      2. Co–clustered mutation pairs (same node)
      3. Separate-lineage mutation pairs (neither ancestor nor descendant)
    """

    # --------------------------------------------------------
    # 1. Build adjacency
    # --------------------------------------------------------
    children = defaultdict(list)
    for src, dst, _ in T.edges(data=True):
        children[src].append(dst)

    # --------------------------------------------------------
    # 2. Precompute descendants for each node
    # --------------------------------------------------------
    def get_descendants(node):
        dq = deque([node])
        seen = set()
        while dq:
            x = dq.popleft()
            for child in children[x]:
                if child not in seen:
                    seen.add(child)
                    dq.append(child)
        return seen

    descendants = {n: get_descendants(n) for n in T.nodes}

    # --------------------------------------------------------
    # 3. Extract SNVs/CNAs
    # --------------------------------------------------------
    if _type == "simulation":
        snv_re = re.compile(r"SNV\d+")
        cna_re = re.compile(r"(Loss|Gain)\s+(\d+)-(\d+)")
        cnloh_re = re.compile(r"(CNLOH)\s+(\d+)-(\d+)")
    elif _type == "LoPhy":
        snv_re = re.compile(r"SNV\d+")
        cna_re = re.compile(r"(Loss|Gain)\s+(\d+)\s+(\d+)",re.IGNORECASE)
        cnloh_re = re.compile(r"(CNLOH)\s+(\d+)\s+(\d+)",re.IGNORECASE)
    elif _type == "COMPASS":
        snv_re = re.compile(r"SNV\d+")
        cna_re = re.compile(
            r"(Loss|Gain)\s+Region(\d+):(\d+)\s+\(chr\d+\)",
            re.IGNORECASE,
        )
        cnloh_re = re.compile(
            r"(CNLOH)\s+Region(\d+):(\d+)\s+\(chr\d+\)",
            re.IGNORECASE,
        )
    elif _type == "SCITE" or _type == "LACE":
        snv_re = re.compile(f"SNV\d+")
        cna_re = re.compile(r"\b\B")
        cnloh_re = re.compile(r"\b\B")
        
    node_snvs = {}
    node_cnas = {}
    node_cnloh = {}

    for node in T.nodes:
        label = T.nodes[node].get("label", "")
        clean = label.replace("REF", "0").replace("ALT", "1")
        node_snvs[node] = snv_re.findall(clean)
        node_cnas[node] = cna_re.findall(clean)
        node_cnloh[node] = cnloh_re.findall(clean)


    # --------------------------------------------------------
    # 4. Collect all mutations across all nodes
    # --------------------------------------------------------
    all_snvs = [(snv, node) for node in T.nodes for snv in node_snvs[node]]
    all_cnas = [(cna, node) for node in T.nodes for cna in node_cnas[node]]
    all_cnloh = [(cnloh, node) for node in T.nodes for cnloh in node_cnloh[node]]
    
    # Flatten to mutation list
    all_mutations = all_snvs + all_cnas + all_cnloh

    # --------------------------------------------------------
    # Output buckets
    # --------------------------------------------------------
    ancestor_descendant = []    # (mut1, mut2)
    coclustered = []            # (mut1, mut2)
    separate_lineage = []       # (mut1, mut2)

    # --------------------------------------------------------
    # 5. Classify pairwise mutation relationships
    # --------------------------------------------------------
    for i in range(len(all_mutations)):
        m1, n1 = all_mutations[i]

        for j in range(i+1, len(all_mutations)):
            m2, n2 = all_mutations[j]

            if n1 == n2:
                coclustered.append((m1, m2))

            elif n2 in descendants[n1]:
                ancestor_descendant.append((m1, m2))

            elif n1 in descendants[n2]:
                ancestor_descendant.append((m2, m1))

            else:
                separate_lineage.append((m1, m2))

    # --------------------------------------------------------
    # Return everything
    # --------------------------------------------------------
    return {
        "ancestor_descendant": ancestor_descendant,
        "coclustered": coclustered,
        "separate_lineage": separate_lineage,
        "node_snvs": node_snvs,
        "node_cnas": node_cnas,
        "descendants": descendants,
        "all_mutations": all_mutations
    }

In [ ]:
import numpy as np
from collections import defaultdict

def canonical(a, b):
    """
    Return an unordered pair in a consistent order,
    works for both strings (SNVs) and tuples (CNAs)
    """
    # Convert everything to a tuple for comparison
    ta = a if isinstance(a, tuple) else (a,)
    tb = b if isinstance(b, tuple) else (b,)
    return (a, b) if ta <= tb else (b, a)


def count_pairs(pairs):
    d = defaultdict(int)
    for a, b in pairs:
        d[(a, b)] += 1
    return d

def compute_prf(gt_pairs, inf_pairs):
    """
    Compute precision, recall, F1 using multiset counts.
    gt_pairs and inf_pairs are lists of (x, y) pairs.
    """

    gt = count_pairs(gt_pairs)
    inf = count_pairs(inf_pairs)

    tp = 0
    total_gt = sum(gt.values())
    total_inf = sum(inf.values())

    for pair, gt_count in gt.items():
        tp += min(gt_count, inf.get(pair, 0))

    if total_gt == 0 or total_inf == 0:
        return (np.nan, np.nan, np.nan)

    precision = tp / total_inf
    recall = tp / total_gt
    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1


In [ ]:
def evaluate_relationships(GT, INF):
    """
    Computes:
        • Mutation-level precision/recall/F1 for:
            SNV–SNV, CNA–CNA, SNV–CNA, CNA–SNV
            across: ancestor-descendant, coclustered, separate-lineage

        • Tree-level unified precision/recall/F1 for:
            ancestor-descendant, coclustered, separate-lineage

        • NEW:
            SNV-only overall metric (all relationships)
            CNA-only overall metric (all relationships)
            Tree-wide overall metric (all mutation types + all relationships)
    """

    results = {}

    # ------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------
    def is_snv(x):
        return isinstance(x, str) and x.startswith("SNV")

    def split_mutation_pairs(pairs):
        """Return: snv_snv, cna_cna, snv_cna, cna_snv"""
        snv_snv, cna_cna, snv_cna, cna_snv = [], [], [], []
        for a, b in pairs:
            A, B = is_snv(a), is_snv(b)
            if A and B:
                snv_snv.append((a, b))
            elif (not A) and (not B):
                cna_cna.append((a, b))
            elif A and (not B):
                snv_cna.append((a, b))
            else:
                cna_snv.append((a, b))
        return snv_snv, cna_cna, snv_cna, cna_snv

    # ------------------------------------------------------------
    # MUTATION-LEVEL METRICS FOR EACH RELATIONSHIP
    # ------------------------------------------------------------
    for rel in ["ancestor_descendant", "coclustered", "separate_lineage"]:

        unordered = (rel in ["coclustered", "separate_lineage"])

        GT_rel = GT[rel]
        INF_rel = INF[rel]

        # Split
        gt_snv_snv, gt_cna_cna, gt_snv_cna, gt_cna_snv = split_mutation_pairs(GT_rel)
        inf_snv_snv, inf_cna_cna, inf_snv_cna, inf_cna_snv = split_mutation_pairs(INF_rel)

        gt_snv_snv = [canonical(a, b) if unordered else (a, b) for a, b in gt_snv_snv]
        inf_snv_snv = [canonical(a, b) if unordered else (a, b) for a, b in inf_snv_snv]
        gt_cna_cna = [canonical(a, b) if unordered else (a, b) for a, b in gt_cna_cna]
        inf_cna_cna = [canonical(a, b) if unordered else (a, b) for a, b in inf_cna_cna]

        # Compute all 4 types
        results[f"SNV_SNV_{rel}"] = compute_prf(gt_snv_snv, inf_snv_snv)
        results[f"CNA_CNA_{rel}"] = compute_prf(gt_cna_cna, inf_cna_cna)
        results[f"SNV_CNA_{rel}"] = compute_prf(gt_snv_cna, inf_snv_cna)
        results[f"CNA_SNV_{rel}"] = compute_prf(gt_cna_snv, inf_cna_snv)

    # ------------------------------------------------------------
    # TREE-LEVEL METRICS for each relationship
    # ------------------------------------------------------------
    for rel in ["ancestor_descendant", "coclustered", "separate_lineage"]:
        unordered = (rel in ["coclustered", "separate_lineage"])
        gt_pairs = [canonical(a, b) if unordered else (a, b) for a, b in GT[rel]]
        inf_pairs = [canonical(a, b) if unordered else (a, b) for a, b in INF[rel]]
        results[f"TREE_{rel}"] = compute_prf(gt_pairs, inf_pairs)

    # ------------------------------------------------------------
    # OVERALL SNV-only (SNV–SNV across all relationships)
    # ------------------------------------------------------------
    all_gt_snv = []
    all_inf_snv = []

    for rel in ["ancestor_descendant", "coclustered", "separate_lineage"]:
        unordered = rel in ["coclustered", "separate_lineage"]

        gt_snv_snv, _, _, _ = split_mutation_pairs(GT[rel])
        inf_snv_snv, _, _, _ = split_mutation_pairs(INF[rel])

        gt_snv_snv = [canonical(a, b) if unordered else (a, b) for a, b in gt_snv_snv]
        inf_snv_snv = [canonical(a, b) if unordered else (a, b) for a, b in inf_snv_snv]

        all_gt_snv.extend(gt_snv_snv)
        all_inf_snv.extend(inf_snv_snv)

    results["SNV_ONLY_OVERALL"] = compute_prf(all_gt_snv, all_inf_snv)

    # ------------------------------------------------------------
    # OVERALL CNA-only (CNA–CNA across all relationships)
    # ------------------------------------------------------------
    all_gt_cna = []
    all_inf_cna = []

    for rel in ["ancestor_descendant", "coclustered", "separate_lineage"]:
        unordered = rel in ["coclustered", "separate_lineage"]

        _, gt_cna_cna, _, _ = split_mutation_pairs(GT[rel])
        _, inf_cna_cna, _, _ = split_mutation_pairs(INF[rel])

        gt_cna_cna = [canonical(a, b) if unordered else (a, b) for a, b in gt_cna_cna]
        inf_cna_cna = [canonical(a, b) if unordered else (a, b) for a, b in inf_cna_cna]

        all_gt_cna.extend(gt_cna_cna)
        all_inf_cna.extend(inf_cna_cna)

    results["CNA_ONLY_OVERALL"] = compute_prf(all_gt_cna, all_inf_cna)

    # ------------------------------------------------------------
    # TREE-WIDE OVERALL (all SNVs and CNAs, all relationships)
    # ------------------------------------------------------------
    all_gt_pairs = []
    all_inf_pairs = []

    for rel in ["ancestor_descendant", "coclustered", "separate_lineage"]:
        unordered = rel in ["coclustered", "separate_lineage"]

        gt_pairs = [canonical(a, b) if unordered else (a, b) for (a, b) in GT[rel]]
        inf_pairs = [canonical(a, b) if unordered else (a, b) for (a, b) in INF[rel]]

        all_gt_pairs.extend(gt_pairs)
        all_inf_pairs.extend(inf_pairs)

    results["TREE_OVERALL"] = compute_prf(all_gt_pairs, all_inf_pairs)

    return results


In [ ]:
def scores_to_df(scores_dict, dataset_name, num_samples, method_name):
    """
    Convert the nested scores dictionary into a long-form DataFrame
    """
    rows = []
    for category, (p, r, f1) in scores_dict.items():
        # Handle case where scores are nan
        p = np.nan if p is None else p
        r = np.nan if r is None else r
        f1 = np.nan if f1 is None else f1

        rows.append({
            "Dataset": dataset_name,
            "Samples": num_samples,
            "Method": method_name,
            "Category": category,
            "Precision": p,
            "Recall": r,
            "F1 Score": f1
        })
    return pd.DataFrame(rows)

In [ ]:
def compute_ancestral_relationship_scores(path, 
                                          samples=[1,2,3,4,5],
                                          include_LoPhy=True,
                                          include_COMPASS=True,
                                          include_SCITE=True,
                                          include_LACE=True):
    from sklearn.metrics import adjusted_rand_score
    
    all_results = []
    for num_samples in samples:
        print(num_samples)
        sim_path = os.path.join(path, f"sims_{num_samples}samples")
        for sim_folder in os.listdir(sim_path):
            sim_directory = Path(os.path.join(sim_path, sim_folder))
            print(sim_folder)
            if sim_directory.is_dir():
                h5ad_fn = os.path.join(sim_directory, "adata.h5ad")
                adata = io.read_h5ad(h5ad_fn)
                
                # load ground truth tree and compute the set of mutation pairwise relationships
                T_true = load_tree(adata, "mutation_tree")
                results_gt = resolve_mutation_relationships(T_true)
                true_cell_assignments = T_true.graph["cell_assignments"]

                # load results for methods if applicable
                if include_LoPhy:
                    _, T_LoPhy = op.io.load_dot(os.path.join(sim_directory, "LoPhy", "out_ml0.gv"), _type="cell_tree")
                    results_LoPhy = resolve_mutation_relationships(T_LoPhy, "LoPhy")
                    scores_LoPhy = evaluate_relationships(results_gt, results_LoPhy)
                    LoPhy_df = scores_to_df(scores_LoPhy, sim_folder, num_samples, "LoPhy")
                    LoPhy_cluster_assignments = np.array(T_LoPhy.graph["cell_assignments"], dtype=int)
                    LoPhy_df["ARI"] = adjusted_rand_score(true_cell_assignments, LoPhy_cluster_assignments)
                    all_results.append(LoPhy_df)

                if include_COMPASS:
                    T_COMPASS = nx.nx_pydot.read_dot(os.path.join(sim_directory, "COMPASS", "out_tree.gv"))
                    results_COMPASS = resolve_mutation_relationships(T_COMPASS, "COMPASS")
                    scores_COMPASS = evaluate_relationships(results_gt, results_COMPASS)
                    COMPASS_df = scores_to_df(scores_COMPASS, sim_folder, num_samples, "COMPASS")
                    COMPASS_cell_assignments_df = pd.read_csv(os.path.join(sim_directory, "COMPASS", "out_cellAssignments.tsv"), header=0, index_col=0, sep="\t")
                    COMPASS_cell_assignments = COMPASS_cell_assignments_df["node"].values
                    COMPASS_df["ARI"] = adjusted_rand_score(true_cell_assignments, COMPASS_cell_assignments)
                    all_results.append(COMPASS_df)

                if include_SCITE:
                    cells = adata.obs.index.to_series().values
                    mapping = {'s%d' % i:cells[i] for i in range(len(cells))}
                    T_SCITE_cell, T_SCITE = op.io.load_dot(os.path.join(sim_directory, "SCITE", "output_ml0.gv"), 
                                               mutations = list(adata.var.index), 
                                               cells = list(adata.obs.index), 
                                               mapping=mapping,
                                               _type="cell_tree")
                    for n in T_SCITE.nodes:
                        T_SCITE.nodes[n]["label"] = n
                    results_SCITE = resolve_mutation_relationships(T_SCITE, "SCITE")
                    scores_SCITE = evaluate_relationships(results_gt, results_SCITE)
                    SCITE_df = scores_to_df(scores_SCITE, sim_folder, num_samples, "SCITE")
                    (_, SCITE_ccm) = op.ul.resolve_genotypes(T_SCITE_cell, adata.X)
                    SCITE_ccm["cluster"] = pd.factorize(
                        [tuple(row) for row in SCITE_ccm.to_numpy()]
                    )[0]
                    SCITE_cluster_assignments = SCITE_ccm["cluster"].values
                    SCITE_df["ARI"]= adjusted_rand_score(true_cell_assignments, SCITE_cluster_assignments)
                    all_results.append(SCITE_df)


                if include_LACE and num_samples > 1:
                    mapping = {'s%d' % i:cells[i] for i in range(len(cells))}
                    T_LACE_cell, T_LACE = op.io.load_dot(os.path.join(sim_directory, "LACE", "tree.gv"),
                                               mutations = list(adata.var.index), 
                                               cells = list(adata.obs.index), 
                                               mapping=mapping,
                                               _type="cell_tree")
                    for n in T_LACE.nodes:
                        T_LACE.nodes[n]["label"] = n
                    results_LACE = resolve_mutation_relationships(T_LACE, "LACE")
                    scores_LACE = evaluate_relationships(results_gt, results_LACE)
                    LACE_df = scores_to_df(scores_LACE, sim_folder, num_samples, "LACE")
                    (_, LACE_ccm) = op.ul.resolve_genotypes(T_LACE_cell, adata.X)
                    LACE_ccm["cluster"] = pd.factorize(
                        [tuple(row) for row in LACE_ccm.to_numpy()]
                    )[0]

                    LACE_cluster_assignments = LACE_ccm["cluster"].values
                    LACE_df["ARI"] = adjusted_rand_score(true_cell_assignments, LACE_cluster_assignments)
                    all_results.append(LACE_df)
            

    results_df = pd.concat(all_results, ignore_index=True)
    return results_df


In [ ]:
def compute_CNA_recall(path, 
                      samples=[1,2,3,4,5]):
    from sklearn.metrics import adjusted_rand_score
    
    all_results = []
    for num_samples in samples:
        print(num_samples)
        sim_path = os.path.join(path, f"sims_{num_samples}samples")
        for sim_folder in os.listdir(sim_path):
            sim_directory = Path(os.path.join(sim_path, sim_folder))
            print(sim_folder)
            if sim_directory.is_dir():
                h5ad_fn = os.path.join(sim_directory, "adata.h5ad")
                adata = io.read_h5ad(h5ad_fn)
                
                # load ground truth tree and compute the set of mutation pairwise relationships
                T_true = load_tree(adata, "mutation_tree")
                results_gt = resolve_mutation_relationships(T_true)

                _, T_LoPhy = op.io.load_dot(os.path.join(sim_directory, "LoPhy", "out_ml0.gv"), _type="cell_tree")
                results_LoPhy = resolve_mutation_relationships(T_LoPhy, "LoPhy")

                T_COMPASS = nx.nx_pydot.read_dot(os.path.join(sim_directory, "COMPASS", "out_tree.gv"))
                results_COMPASS = resolve_mutation_relationships(T_COMPASS, "COMPASS")
                   
                true_gains = set([x[0] for x in results_gt["all_mutations"] if x[0][0].lower() == "gain"])
                true_losses = set([x[0] for x in results_gt["all_mutations"] if x[0][0].lower() == "loss"])
                true_cnloh = set([x[0] for x in results_gt["all_mutations"] if x[0][0].lower() == "cnloh"])

                LoPhy_gains = set([x[0] for x in results_LoPhy["all_mutations"] if x[0][0].lower() == "gain"])
                LoPhy_losses = set([x[0]for x in results_LoPhy["all_mutations"] if x[0][0].lower() == "loss"])
                LoPhy_cnloh = set([x[0] for x in results_LoPhy["all_mutations"] if x[0][0].lower() == "cnloh"])
                
                COMPASS_gains = set([x[0] for x in results_COMPASS["all_mutations"] if x[0][0].lower() == "gain"])
                COMPASS_losses = set([x[0] for x in results_COMPASS["all_mutations"] if x[0][0].lower() == "loss"])
                COMPASS_cnloh = set([x[0] for x in results_COMPASS["all_mutations"] if x[0][0].lower() == "cnloh"])
                
#                 print("Ground Truth")
#                 op.pl.show_tree(T_true)
#                 print(true_gains)
#                 print(true_losses)
#                 print(true_cnloh)
#                 print("\n")
#                 print("LoPhy")
#                 op.pl.show_tree(T_LoPhy)
#                 print(LoPhy_gains)
#                 print(LoPhy_losses)
#                 print(LoPhy_cnloh)
#                 print("\n")
#                 print("COMPASS")
#                 op.pl.show_tree(T_COMPASS)
#                 print(COMPASS_gains)
#                 print(COMPASS_losses)
#                 print(COMPASS_cnloh)
                all_results.append([sim_folder, 
                                    num_samples, 
                                    "LoPhy", 
                                    len(true_gains),
                                    len(true_losses),
                                    len(true_cnloh),
                                    len(true_gains & LoPhy_gains),
                                    len(true_losses & LoPhy_losses), 
                                    len(true_cnloh & LoPhy_cnloh), 
                                    len(LoPhy_gains - true_gains),
                                    len(LoPhy_losses - true_losses), 
                                    len(LoPhy_cnloh - true_cnloh)
                                   ])
                all_results.append([sim_folder, 
                                    num_samples, 
                                    "COMPASS", 
                                    len(true_gains),
                                    len(true_losses),
                                    len(true_cnloh),
                                    len(true_gains & COMPASS_gains),
                                    len(true_losses & COMPASS_losses), 
                                    len(true_cnloh & COMPASS_cnloh), 
                                    len(COMPASS_gains - true_gains),
                                    len(COMPASS_losses - true_losses), 
                                    len(COMPASS_cnloh - true_cnloh)
                                   ])
                
    results_df = pd.DataFrame(all_results, 
                              columns=["simulation", "samples", "Method", "# Gains", "# Loss", "# CNLOH", "# Correct Gains", "# Correct Losses", "# Correct CNLOH", "# Incorrect Gains", "# Incorrect Losses", "# Incorrect CNLOH"])
    return results_df


In [ ]:
def plot_LoPhy_vs_ground_truth_trees(path, 
                                     samples=[1,2,3,4,5]):

    all_results = []
    for num_samples in samples:
        print(num_samples)
        sim_path = os.path.join(path, f"sims_{num_samples}samples")
        for sim_folder in os.listdir(sim_path):
            sim_directory = Path(os.path.join(sim_path, sim_folder))
            print(sim_folder)
            if sim_directory.is_dir():
                h5ad_fn = os.path.join(sim_directory, "adata.h5ad")
                adata = io.read_h5ad(h5ad_fn)
                
                # load ground truth tree and compute the set of mutation pairwise relationships
                print("\n ----- groudn truth -------")
                T_true = load_tree(adata, "mutation_tree")
                op.pl.show_tree(T_true)

                # load results for methods if applicable
                print("\n ----- LoPhy -------")
                _, T_LoPhy = op.io.load_dot(os.path.join(sim_directory, "LoPhy", "out_ml0.gv"), _type="cell_tree")
                op.pl.show_tree(T_LoPhy)


                # load results for methods if applicable
                print("\n ----- COMPASS -------")
                T_COMPASS = nx.nx_pydot.read_dot(os.path.join(sim_directory, "COMPASS", "out_tree.gv"))
                op.pl.show_tree(T_COMPASS)

In [ ]:
def plot_ancestral_relationship_metrics1(results_df,
                                         palette="deep",
                                         hue_order=["LoPhy", "COMPASS", "SCITE", "LACE"],
                                         save_path=os.getcwd(), 
                                         file_name="tree_correspondence_metrics1.svg"):

    # Categories corresponding to the 3 rows
    row_categories = [
        "SNV_ONLY_OVERALL",
        "CNA_ONLY_OVERALL",
        "TREE_OVERALL"
    ]

    metrics = ["Precision", "Recall", "F1 Score"]

    fig, axes = plt.subplots(3, 3, figsize=(16, 16))

    for row_idx, cat in enumerate(row_categories):
        df_cat = results_df[results_df["Category"] == cat]

        for col_idx, metric in enumerate(metrics):
            ax = axes[row_idx, col_idx]

            sns.boxplot(
                data=df_cat,
                x="Samples",
                y=metric,
                hue="Method",
                palette=palette,
                hue_order=hue_order,
                ax=ax
            )

            ax.set_title(f"{cat} — {metric}")
            ax.set_xlabel("Samples")
            ax.set_ylabel(metric)
            ax.legend_.remove()  # We'll add a shared legend later

    # --- Shared legend ---
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=2,
        frameon=False
    )

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    if len(save_path) > 0:
        fig.savefig(os.path.join(save_path, file_name))
    plt.show()


In [ ]:
def plot_ancestral_relationship_metrics2(results_df,
                                         hue_order=["LoPhy", "COMPASS", "SCITE", "LACE"],
                                         save_path=os.getcwd(), 
                                         file_name="tree_correspondence_metrics2.svg"):

    # Categories corresponding to the 3 rows
    row_categories = [
        "SNV_SNV_ancestor_descendant",
        "SNV_SNV_coclustered",
        "SNV_SNV_separate_lineage"
    ]

    metrics = ["Precision", "Recall", "F1 Score"]

    fig, axes = plt.subplots(3, 3, figsize=(16, 16))

    for row_idx, cat in enumerate(row_categories):
        df_cat = results_df[results_df["Category"] == cat]

        for col_idx, metric in enumerate(metrics):
            ax = axes[row_idx, col_idx]

            sns.boxplot(
                data=df_cat,
                x="Samples",
                y=metric,
                hue="Method",
                palette="deep",
                hue_order=hue_order,
                ax=ax
            )

            ax.set_title(f"{cat} — {metric}")
            ax.set_xlabel("Samples")
            ax.set_ylabel(metric)
            ax.legend_.remove()  # We'll add a shared legend later

    # --- Shared legend ---
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=2,
        frameon=False
    )

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    if len(save_path) > 0:
        fig.savefig(os.path.join(save_path, file_name))
    plt.show()



In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns


def plot_ancestral_relationship_metrics3(
    results_df,
    partition_col,
    metric="F1 Score",
    categories=(
        "SNV_ONLY_OVERALL",
        "CNA_ONLY_OVERALL",
        "TREE_OVERALL",
    ),
    sample_col="Samples",
    hue_col="Method",
    hue_order=("LoPhy", "COMPASS", "SCITE", "LACE"),
    palette="deep",
    save_path="",
    file_name="ancestral_relationship_metrics3.svg",
):

    sample_levels = sorted(results_df[sample_col].unique())
    partition_levels = sorted(results_df[partition_col].dropna().unique())

    fig, axes = plt.subplots(
        len(categories),
        len(sample_levels),
        figsize=(4 * len(sample_levels), 4 * len(categories)),
        sharex=True,
        sharey="row",
    )

    if len(categories) == 1:
        axes = axes[None, :]
    if len(sample_levels) == 1:
        axes = axes[:, None]

    handles = labels = None

    for i, category in enumerate(categories):

        df_cat = results_df[results_df["Category"] == category]

        for j, sample in enumerate(sample_levels):

            ax = axes[i, j]

            subset = df_cat[df_cat[sample_col] == sample]

            sns.boxplot(
                data=subset,
                x=partition_col,
                y=metric,
                hue=hue_col,
                hue_order=hue_order,
                order=partition_levels,
                palette=palette,
                ax=ax,
            )

            if handles is None:
                handles, labels = ax.get_legend_handles_labels()

            if ax.get_legend() is not None:
                ax.get_legend().remove()

            if i == 0:
                ax.set_title(f"{sample} samples")

            if j == 0:
                ax.set_ylabel(category.replace("_", "\n"))
            else:
                ax.set_ylabel("")

            if i == len(categories) - 1:
                ax.set_xlabel(partition_col)
            else:
                ax.set_xlabel("")

    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=len(labels),
        frameon=False,
    )

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    if save_path:
        fig.savefig(
            os.path.join(save_path, file_name),
            bbox_inches="tight",
        )

    plt.show()

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns


def plot_ari_result(
    ARI_results_df,
    partition_col="Samples",
    sample_col="Samples",
    palette="deep",
    hue_order=["LoPhy", "COMPASS", "SCITE", "LACE"],
    save_path="",
    figure_name="ARI_metric.svg",
):

    # Keep only rows with ARI
    df = ARI_results_df.dropna(subset=["ARI"]).copy()

    # Determine methods actually present
    methods_present = list(df["Method"].unique())

    if hue_order is None:
        hue_order = methods_present
    else:
        # Keep only methods that exist in the dataframe
        hue_order = [m for m in hue_order if m in methods_present]

    sample_levels = sorted(df[sample_col].unique())
    partition_levels = sorted(df[partition_col].unique())

    fig, axes = plt.subplots(
        1,
        len(sample_levels),
        figsize=(4 * len(sample_levels), 4),
        sharey=True,
    )

    if len(sample_levels) == 1:
        axes = [axes]

    handles = labels = None

    for j, sample in enumerate(sample_levels):

        ax = axes[j]

        subset = df[df[sample_col] == sample]

        sns.boxplot(
            data=subset,
            x=partition_col,
            y="ARI",
            hue="Method",
            hue_order=hue_order,
            order=partition_levels,
            palette=palette,
            ax=ax,
        )

        ax.set_title(f"{sample} samples")
        ax.set_xlabel(partition_col)

        if j == 0:
            ax.set_ylabel("ARI")
        else:
            ax.set_ylabel("")

        if ax.get_legend() is not None:
            if handles is None:
                handles, labels = ax.get_legend_handles_labels()
            ax.get_legend().remove()

    if handles is not None:
        fig.legend(
            handles,
            labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 1.03),
            ncol=len(labels),
            frameon=False,
        )

    plt.tight_layout(rect=[0, 0, 1, 0.95])

    if save_path and figure_name:
        fig.savefig(
            os.path.join(save_path, figure_name),
            bbox_inches="tight",
        )

    plt.show()

## Simulations

In [ ]:
path = os.path.join(os.getcwd(), "simulations")

In [ ]:
simulations_results_df = compute_ancestral_relationship_scores(path, samples=[2,3,4,5])

In [ ]:
simulation_cna_results = compute_CNA_recall(path)

In [ ]:
simulation_cna_results

In [ ]:
import pandas as pd

results = []

for num_samples in simulation_cna_results["samples"].unique():
    for method in simulation_cna_results["Method"].unique():

        subset = simulation_cna_results[
            (simulation_cna_results["samples"] == num_samples) &
            (simulation_cna_results["Method"] == method)
        ]

        for cna_type, correct_col, incorrect_col, total_col in [
            ("Gain", "# Correct Gains", "# Incorrect Gains", "# Gains"),
            ("Loss", "# Correct Losses", "# Incorrect Losses", "# Loss"),
            ("CNLOH", "# Correct CNLOH", "# Incorrect CNLOH", "# CNLOH")
        ]:

            tp = subset[correct_col].sum()
            fp = subset[incorrect_col].sum()

            # divide by 2 because your dataframe has both LoPhy and COMPASS rows
            # and the ground truth is duplicated for each method
            total = (
                simulation_cna_results.loc[
                    simulation_cna_results["samples"] == num_samples,
                    total_col
                ].sum() / 2
            )

            fn = total - tp

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / total if total > 0 else 0

            results.append([
                num_samples,
                method,
                cna_type,
                tp,
                fp,
                total,
                precision,
                recall
            ])

cna_pr_df = pd.DataFrame(
    results,
    columns=[
        "Samples",
        "Method",
        "CNA type",
        "TP",
        "FP",
        "Total true events",
        "Precision",
        "Recall"
    ]
)

cna_pr_df

In [ ]:
simulation_cna_results.loc[
    (simulation_cna_results["Method"] == "COMPASS") & (simulation_cna_results["samples"] == num_samples),
    "# Gains"]

In [ ]:
    LoPhy_gains = simulation_cna_results.loc[
        (simulation_cna_results["Method"] == "LoPhy") & (simulation_cna_results["samples"] == 1),
        "# Correct Gains"].values.sum()

In [ ]:
LoPhy_gains

In [ ]:
simulations_results_df.to_csv(os.path.join(os.getcwd(), "simulation_results", "simulations_metrics2_fig2.csv"))

In [ ]:
samples1_simulations_results_df = compute_ancestral_relationship_scores(path, samples=[1])

In [ ]:
samples1_simulations_results_df

In [ ]:
plot_ancestral_relationship_metrics1(samples1_simulations_results_df,
                                    save_path=os.path.join(os.getcwd(), "..", "paper", "sim_results", "simulations"), 
                                    file_name="1sample_tree_correspondence.svg")

In [ ]:
plot_ancestral_relationship_metrics1(simulations_results_df,
                                    save_path=os.path.join(os.getcwd(), "..", "paper", "sim_results", "simulations"), 
                                    file_name="tree_correspondence.svg")

In [ ]:
plot_ancestral_relationship_metrics2(simulations_results_df,
                                    save_path=os.path.join(os.getcwd(), "..", "paper", "sim_results", "simulations"), 
                                    file_name="snv_tree_correspondence.svg")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots()

sns.boxplot(
    data=simulations_results_df,
    x="Samples",
    y="ARI",
    hue="Method",
    palette="deep",
    hue_order=["LoPhy", "COMPASS", "SCITE", "LACE"],
    ax=ax
)

# Remove the axes legend and create a shared figure legend
handles, labels = ax.get_legend_handles_labels()
ax.get_legend().remove()

fig.legend(
    handles,
    labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.05),
    ncol=len(labels),
    frameon=False
)

plt.tight_layout(rect=[0, 0, 1, 0.98])  # Leave space for legend
fig.savefig(os.path.join(os.getcwd(), "..", "paper", "sim_results", "simulations", "ARI_metric.svg"), format="svg")
plt.show()

## Uniform coverage

In [ ]:
uniform_coverage_simulations_COMPASS = os.path.join(os.getcwd(), "uniform_coverage_simulations")
uniform_coverage_simulations_LoPhy = os.path.join(os.getcwd(), "uniform_coverage_simulations_LoPhy")
uniform_coverage_simulations_LoPhy_3chains = os.path.join(os.getcwd(), "uniform_coverage_simulations_LoPhy_3chains")


COMPASS_uniform_coverage_df = compute_ancestral_relationship_scores(uniform_coverage_simulations_COMPASS, 
                                                                 samples=[1,2,3,4,5],
                                                                include_LoPhy=False,
                                                                include_COMPASS=True,
                                                                include_SCITE=False,
                                                                include_LACE=False)
LoPhy_uniform_coverage_df = compute_ancestral_relationship_scores(uniform_coverage_simulations_LoPhy, 
                                                                samples=[1,2,3,4,5],
                                                                include_LoPhy=True,
                                                                include_COMPASS=False,
                                                                include_SCITE=False,
                                                                include_LACE=False)
LoPhy_3chains_uniform_coverage_df = compute_ancestral_relationship_scores(uniform_coverage_simulations_LoPhy_3chains, 
                                                                samples=[1,2,3,4,5],
                                                                include_LoPhy=True,
                                                                include_COMPASS=False,
                                                                include_SCITE=False,
                                                                include_LACE=False)

In [ ]:
LoPhy_3chains_uniform_coverage_df["Method"] = LoPhy_3chains_uniform_coverage_df["Method"].replace({"LoPhy": "LoPhy (i=3)"})

In [ ]:
uniform_coverage_results_df = pd.concat([COMPASS_uniform_coverage_df, LoPhy_uniform_coverage_df, LoPhy_3chains_uniform_coverage_df]).reset_index(drop=True)

In [ ]:
uniform_coverage_results_df.to_csv(os.path.join(os.getcwd(), "simulation_results", "uniform_coverage_metrics2.csv"))

In [ ]:
uniform_coverage_results_df

In [ ]:
uniform_coverage_results_df[uniform_coverage_results_df["Category"] == "TREE_OVERALL"].groupby(["Method", "Samples"])[["Precision", "Recall", "F1 Score", "ARI"]].mean()

In [ ]:
deep_colors = sns.color_palette("deep")
plot_ancestral_relationship_metrics1(uniform_coverage_results_df,
                                     palette=[deep_colors[0], deep_colors[4], deep_colors[1]], 
                                     hue_order=["LoPhy", "LoPhy (i=3)", "COMPASS"],
                                     save_path=os.path.join(os.getcwd(), "..", "paper", "sim_results", "uniform_coverage"), 
                                     file_name="uniform_coverage_tree_correspondence.svg")

In [ ]:
plot_ari_result(uniform_coverage_results_df,
                                    palette=[deep_colors[0], deep_colors[4], deep_colors[1]], 
                                    hue_order=["LoPhy", "LoPhy (i=3)", "COMPASS"],
                                    save_path=os.path.join(os.getcwd(), "..", "paper", "sim_results", "uniform_coverage"), 
                                    figure_name="uniform_coverage_ARI.svg")

## Doublets

In [ ]:
doublets_0_01_path = os.path.join(os.getcwd(), "doublets_0.01_simulations")
doublets_0_1_path = os.path.join(os.getcwd(), "doublets_0.1_simulations")
doublets_0_5_path = os.path.join(os.getcwd(), "doublets_0.5_simulations")

In [ ]:
doublets_0_01_results_df = compute_ancestral_relationship_scores(doublets_0_01_path, 
                                                                 samples=[1,2,3,4,5],
                                                                include_LoPhy=True,
                                                                include_COMPASS=True,
                                                                include_SCITE=True,
                                                                include_LACE=True)
doublets_0_1_results_df = compute_ancestral_relationship_scores(doublets_0_1_path, 
                                                                samples=[1,2,3,4,5],
                                                                include_LoPhy=True,
                                                                include_COMPASS=True,
                                                                include_SCITE=True,
                                                                include_LACE=True)
doublets_0_5_results_df = compute_ancestral_relationship_scores(doublets_0_5_path, 
                                                                samples=[1,2,3,4,5],
                                                                include_LoPhy=True,
                                                                include_COMPASS=True,
                                                                include_SCITE=True,
                                                                include_LACE=True)

In [ ]:
doublets_0_01_results_df["doublet rate"] = 0.01
doublets_0_1_results_df["doublet rate"] = 0.1
doublets_0_5_results_df["doublet rate"] = 0.5

doublets_results_df = pd.concat(
    [
        doublets_0_01_results_df,
        doublets_0_1_results_df,
        doublets_0_5_results_df,
    ],
    ignore_index=True,
)

In [ ]:
plot_ancestral_relationship_metrics3(doublets_results_df, 
                                     partition_col="doublet rate",
                                     hue_order=["LoPhy", "COMPASS", "SCITE", "LACE"],
                                     save_path=os.path.join(os.getcwd(), "..", "paper", "sim_results", "doublets"),
                                     file_name="doublets_ancestral_relationship_metrics.svg")

In [ ]:
plot_ari_result(doublets_results_df,
                partition_col="doublet rate",
                hue_order=["LoPhy", "COMPASS", "SCITE", "LACE"],
                save_path=os.path.join(os.getcwd(), "..", "paper", "sim_results", "doublets"),
                figure_name="doublets_ARI.svg")

## Heterozygous precision

In [ ]:
heterozygous_precision_2_0_path = os.path.join(os.getcwd(), "het_precision_2.0_simulations")
heterozygous_precision_4_0_path = os.path.join(os.getcwd(), "simulations")
heterozygous_precision_8_0_path = os.path.join(os.getcwd(), "het_precision_8.0_simulations")

In [ ]:
heterozygous_precision_2_0_results_df = compute_ancestral_relationship_scores(heterozygous_precision_2_0_path, 
                                                                 samples=[1,2,3,4,5],
                                                                include_LoPhy=True,
                                                                include_COMPASS=False,
                                                                include_SCITE=False,
                                                                include_LACE=False)
heterozygous_precision_4_0_results_df = compute_ancestral_relationship_scores(heterozygous_precision_4_0_path, 
                                                                samples=[1,2,3,4,5],
                                                                include_LoPhy=True,
                                                                include_COMPASS=False,
                                                                include_SCITE=False,
                                                                include_LACE=False)
heterozygous_precision_8_0_results_df = compute_ancestral_relationship_scores(heterozygous_precision_8_0_path, 
                                                                samples=[1,2,3,4,5],
                                                                include_LoPhy=True,
                                                                include_COMPASS=False,
                                                                include_SCITE=False,
                                                                include_LACE=False)

In [ ]:
heterozygous_precision_2_0_results_df["heterozygous precision"] = 2.0
heterozygous_precision_4_0_results_df["heterozygous precision"] = 4.0
heterozygous_precision_8_0_results_df["heterozygous precision"] = 8.0

heterozygous_precision_df = pd.concat(
    [
        heterozygous_precision_2_0_results_df,
        heterozygous_precision_4_0_results_df,
        heterozygous_precision_8_0_results_df,
    ],
    ignore_index=True,
)

In [ ]:
heterozygous_precision_df

In [ ]:
plot_ancestral_relationship_metrics3(heterozygous_precision_df, 
                                     "heterozygous precision",
                                     save_path=os.path.join(os.getcwd(), "..", "paper", "sim_results", "heterozygous_precision"),
                                     file_name="heterozygous_precision_ancestral_relationship_metrics.svg")

In [ ]:
plot_ari_result(heterozygous_precision_df,
                partition_col="heterozygous precision",
                save_path=os.path.join(os.getcwd(), "..", "paper", "sim_results", "heterozygous_precision"),
                figure_name="heterozygous_precision_ARI.svg")

## Homozygous precision

In [ ]:
homozygous_precision_8_0_path = os.path.join(os.getcwd(), "hom_precision_8.0_simulations")
homozygous_precision_15_0_path = os.path.join(os.getcwd(), "simulations")
homozygous_precision_50_0_path = os.path.join(os.getcwd(), "hom_precision_50.0_simulations")

In [ ]:
homozygous_precision_8_0_results_df = compute_ancestral_relationship_scores(homozygous_precision_8_0_path, 
                                                                 samples=[1,2,3,4,5],
                                                                include_LoPhy=True,
                                                                include_COMPASS=False,
                                                                include_SCITE=False,
                                                                include_LACE=False)
homozygous_precision_15_0_results_df = compute_ancestral_relationship_scores(homozygous_precision_15_0_path, 
                                                                samples=[1,2,3,4,5],
                                                                include_LoPhy=True,
                                                                include_COMPASS=False,
                                                                include_SCITE=False,
                                                                include_LACE=False)
homozygous_precision_50_0_results_df = compute_ancestral_relationship_scores(homozygous_precision_50_0_path, 
                                                                samples=[1,2,3,4,5],
                                                                include_LoPhy=True,
                                                                include_COMPASS=False,
                                                                include_SCITE=False,
                                                                include_LACE=False)

In [ ]:
homozygous_precision_8_0_results_df["homozygous precision"] = 8.0
homozygous_precision_15_0_results_df["homozygous precision"] = 15.0
homozygous_precision_50_0_results_df["homozygous precision"] = 50.0

homozygous_precision_df = pd.concat(
    [
        homozygous_precision_8_0_results_df,
        homozygous_precision_15_0_results_df,
        homozygous_precision_50_0_results_df,
    ],
    ignore_index=True,
)

In [ ]:
plot_ancestral_relationship_metrics3(homozygous_precision_df, 
                                     "homozygous precision",
                                     save_path=os.path.join(os.getcwd(), "..", "paper", "sim_results", "homozygous_precision"),
                                     file_name="homozygous_precision_ancestral_relationship_metrics.svg")

In [ ]:
plot_ari_result(homozygous_precision_df,
                partition_col="homozygous precision",
                save_path=os.path.join(os.getcwd(), "..", "paper", "sim_results", "homozygous_precision"),
                figure_name="homozygous_precision_ARI.svg")

## Random restarts

In [ ]:
random_restarts_1_path = os.path.join(os.getcwd(), "simulations")
random_restarts_3_path = os.path.join(os.getcwd(), "simulations_main_3restarts")
random_restarts_5_path = os.path.join(os.getcwd(), "simulations_main_5restarts")

In [ ]:
random_restarts_1_results_df = compute_ancestral_relationship_scores(random_restarts_1_path, 
                                                                 samples=[1,2,3,4,5],
                                                                include_LoPhy=True,
                                                                include_COMPASS=False,
                                                                include_SCITE=False,
                                                                include_LACE=False)
random_restarts_3_results_df = compute_ancestral_relationship_scores(random_restarts_3_path, 
                                                                samples=[1,2,3,4,5],
                                                                include_LoPhy=True,
                                                                include_COMPASS=False,
                                                                include_SCITE=False,
                                                                include_LACE=False)
random_restarts_5results_df = compute_ancestral_relationship_scores(random_restarts_5_path, 
                                                                samples=[1,2,3,4,5],
                                                                include_LoPhy=True,
                                                                include_COMPASS=False,
                                                                include_SCITE=False,
                                                                include_LACE=False)

In [ ]:
random_restarts_1_results_df["Random restarts"] = 1
random_restarts_3_results_df["Random restarts"] = 3
random_restarts_5results_df["Random restarts"] = 5

random_restarts_df = pd.concat(
    [
        random_restarts_1_results_df,
        random_restarts_3_results_df,
        random_restarts_5results_df,
    ],
    ignore_index=True,
)

In [ ]:
plot_ancestral_relationship_metrics3(random_restarts_df, 
                                     "Random restarts",
                                     save_path=os.path.join(os.getcwd(), "..", "paper", "sim_results", "random_restarts"),
                                     file_name="random_restarts_ancestral_relationship_metrics.svg")

In [ ]:
plot_ari_result(random_restarts_df,
                partition_col="Random restarts",
                save_path=os.path.join(os.getcwd(), "..", "paper", "sim_results", "random_restarts"),
                figure_name="random_restarts_ARI.svg")

## Max cn = 4, 5

In [ ]:
max_cn_4_path = os.path.join(os.getcwd(), "simulations_max_cn_4")
max_cn_5_path = os.path.join(os.getcwd(), "simulations_max_cn_5")

In [ ]:
max_cn_4_results_df = compute_ancestral_relationship_scores(max_cn_4_path, 
                                                                 samples=[1,2,3,4,5],
                                                                include_LoPhy=True,
                                                                include_COMPASS=False,
                                                                include_SCITE=False,
                                                                include_LACE=False)
max_cn_5_results_df = compute_ancestral_relationship_scores(max_cn_5_path, 
                                                                samples=[1,2,3,4,5],
                                                                include_LoPhy=True,
                                                                include_COMPASS=False,
                                                                include_SCITE=False,
                                                                include_LACE=False)

In [ ]:
max_cn_4_results_df["Max copy number"] = 4
max_cn_5_results_df["Max copy number"] = 5

max_cn_df = pd.concat(
    [
        max_cn_4_results_df,
        max_cn_5_results_df,
    ],
    ignore_index=True,
)

In [ ]:
plot_ancestral_relationship_metrics3(max_cn_df, 
                                     "Max copy number",
                                     save_path=os.path.join(os.getcwd(), "..", "paper", "sim_results", "max_cn"),
                                     file_name="max_cn_ancestral_relationship_metrics.svg")

In [ ]:
plot_ari_result(max_cn_df,
                partition_col="Max copy number",
                save_path=os.path.join(os.getcwd(), "..", "paper", "sim_results", "max_cn"),
                figure_name="max_cn_ARI.svg")

In [ ]:
plot_LoPhy_vs_ground_truth_trees(max_cn_4_path)

In [ ]:
plot_LoPhy_vs_ground_truth_trees(max_cn_5_path)

In [ ]:
def compute_ancestral_relationship_scores1(path, 
                                          samples=[1,2,3,4,5],
                                          include_LoPhy=True,
                                          include_COMPASS=True):
    from sklearn.metrics import adjusted_rand_score
    
    all_results = []
    for num_samples in samples:
        print(num_samples)
        sim_path = os.path.join(path, f"sims_{num_samples}samples")
        for sim_folder in os.listdir(sim_path):
            sim_directory = Path(os.path.join(sim_path, sim_folder))
            print(sim_folder)
            if sim_directory.is_dir():
                h5ad_fn = os.path.join(sim_directory, "adata.h5ad")
                adata = io.read_h5ad(h5ad_fn)
                
                # load ground truth tree and com pute the set of mutation pairwise relationships
                T_true = load_tree(adata, "mutation_tree")
                true_snvs, true_cnas = resolve_mutation_relationships1(T_true)

                # load results for methods if applicable
                if include_LoPhy:
                    _, T_LoPhy = op.io.load_dot(os.path.join(sim_directory, "LoPhy", "out_ml0.gv"), _type="cell_tree")
                    LoPhy_snvs, LoPhy_cnas, LoPhy_cnloh = resolve_mutation_relationships1(T_LoPhy, "LoPhy")


                if include_COMPASS:
                    T_COMPASS = nx.nx_pydot.read_dot(os.path.join(sim_directory, "COMPASS", "out_tree.gv"))
                    COMPASS_snvs, COMPASS_cnas, COMPASS_cnloh = resolve_mutation_relationships1(T_COMPASS, "COMPASS")
                GT_gains = set([cna for cna in true_cnas if cna[0] == "Gain"])
                LoPhy_gains = set([cna for cna in LoPhy_cnas if cna[0] == "Gain"])
                COMPASS_gains = set([cna for cna in COMPASS_cnas if cna[0] == "Gain"])

            print(true_cnas, LoPhy_cnas, COMPASS_cnas)    
        break
    return results_df


In [ ]:
 compute_ancestral_relationship_scores1(os.path.join(os.getcwd(), "simulations")) 

In [ ]:
single_timepoint_results = []
path = os.path.join(os.getcwd(), "simulations_main")
for num_samples in range(1,2):
    print(num_samples)
    sim_path = os.path.join(path, f"sims_{num_samples}samples")
    for sim_folder in os.listdir(sim_path):
        sim_directory = Path(os.path.join(sim_path, sim_folder))
        print(sim_folder)
        if sim_directory.is_dir():
            h5ad_fn = os.path.join(sim_directory, "adata.h5ad")
            adata = io.read_h5ad(h5ad_fn)
            T_true = load_tree(adata, "mutation_tree")
            _, T_LoPhy = op.io.load_dot(os.path.join(sim_directory, "LoPhy", "out_ml0.gv"), _type="cell_tree")
            T_COMPASS = nx.nx_pydot.read_dot(os.path.join(sim_directory, "COMPASS", "out_tree.gv"))
            
            cells = adata.obs.index.to_series().values
            mapping = {'s%d' % i:cells[i] for i in range(len(cells))}
            _, T_SCITE = op.io.load_dot(os.path.join(sim_directory, "SCITE", "output_ml0.gv"), 
                                       mutations = list(adata.var.index), 
                                       cells = list(adata.obs.index), 
                                       mapping=mapping,
                                       _type="cell_tree")
            for n in T_SCITE.nodes:
                T_SCITE.nodes[n]["label"] = n
            
            results_gt = resolve_mutation_relationships(T_true)
            results_COMPASS = resolve_mutation_relationships(T_COMPASS, "COMPASS")
            results_LoPhy = resolve_mutation_relationships(T_LoPhy, "LoPhy")
            results_SCITE = resolve_mutation_relationships(T_SCITE, "SCITE")

            scores_COMPASS = evaluate_relationships(results_gt, results_COMPASS)
            scores_LoPhy = evaluate_relationships(results_gt, results_LoPhy)
            scores_SCITE = evaluate_relationships(results_gt, results_SCITE)
                
            single_timepoint_results.append(scores_to_df(scores_COMPASS, sim_folder, num_samples, "COMPASS"))
            single_timepoint_results.append(scores_to_df(scores_LoPhy, sim_folder, num_samples, "LoPhy"))
            single_timepoint_results.append(scores_to_df(scores_SCITE, sim_folder, num_samples, "SCITE"))
single_timepoint_results_df = pd.concat(single_timepoint_results, ignore_index=True)


In [ ]:
all_results = []
for num_samples in range(2,6):
    print(num_samples)
    sim_path = os.path.join(path, f"sims_{num_samples}samples")
    for sim_folder in os.listdir(sim_path):
        sim_directory = Path(os.path.join(sim_path, sim_folder))
        print(sim_folder)
        if sim_directory.is_dir():
            h5ad_fn = os.path.join(sim_directory, "adata.h5ad")
            adata = io.read_h5ad(h5ad_fn)
            T_true = load_tree(adata, "mutation_tree")
            _, T_LoPhy = op.io.load_dot(os.path.join(sim_directory, "LoPhy", "out_ml0.gv"), _type="cell_tree")
            T_COMPASS = nx.nx_pydot.read_dot(os.path.join(sim_directory, "COMPASS", "out_tree.gv"))
            
            cells = adata.obs.index.to_series().values
            mapping = {'s%d' % i:cells[i] for i in range(len(cells))}
            _, T_SCITE = op.io.load_dot(os.path.join(sim_directory, "SCITE", "output_ml0.gv"), 
                                       mutations = list(adata.var.index), 
                                       cells = list(adata.obs.index), 
                                       mapping=mapping,
                                       _type="cell_tree")
            for n in T_SCITE.nodes:
                T_SCITE.nodes[n]["label"] = n
            
            mapping = {'s%d' % i:cells[i] for i in range(len(cells))}
            _, T_LACE = op.io.load_dot(os.path.join(sim_directory, "LACE", "tree.gv"),
                                       mutations = list(adata.var.index), 
                                       cells = list(adata.obs.index), 
                                       mapping=mapping,
                                       _type="cell_tree")
            for n in T_LACE.nodes:
                T_LACE.nodes[n]["label"] = n

            results_gt = resolve_mutation_relationships(T_true)
            results_COMPASS = resolve_mutation_relationships(T_COMPASS, "COMPASS")
            results_LoPhy = resolve_mutation_relationships(T_LoPhy, "LoPhy")
            results_SCITE = resolve_mutation_relationships(T_SCITE, "SCITE")
            results_LACE = resolve_mutation_relationships(T_LACE, "LACE")

            scores_COMPASS = evaluate_relationships(results_gt, results_COMPASS)
            scores_LoPhy = evaluate_relationships(results_gt, results_LoPhy)
            scores_SCITE = evaluate_relationships(results_gt, results_SCITE)
            scores_LACE = evaluate_relationships(results_gt, results_LACE)
                
            all_results.append(scores_to_df(scores_COMPASS, sim_folder, num_samples, "COMPASS"))
            all_results.append(scores_to_df(scores_LoPhy, sim_folder, num_samples, "LoPhy"))
            all_results.append(scores_to_df(scores_SCITE, sim_folder, num_samples, "SCITE"))
            all_results.append(scores_to_df(scores_LACE, sim_folder, num_samples, "LACE"))
results_df = pd.concat(all_results, ignore_index=True)


In [ ]:
longitudinal_sim_results = results_df[results_df.Samples >= 2]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Categories corresponding to the 3 rows
row_categories = [
    "SNV_ONLY_OVERALL",
    "CNA_ONLY_OVERALL",
    "TREE_OVERALL"
]

metrics = ["Precision", "Recall", "F1 Score"]

fig, axes = plt.subplots(3, 3, figsize=(16, 16))

for row_idx, cat in enumerate(row_categories):
    df_cat = longitudinal_sim_results[longitudinal_sim_results["Category"] == cat]

    for col_idx, metric in enumerate(metrics):
        ax = axes[row_idx, col_idx]

        sns.boxplot(
            data=df_cat,
            x="Samples",
            y=metric,
            hue="Method",
            palette="deep",
            hue_order=["LoPhy", "COMPASS", "SCITE", "LACE"],
            ax=ax
        )

        ax.set_title(f"{cat} — {metric}")
        ax.set_xlabel("Samples")
        ax.set_ylabel(metric)
        ax.legend_.remove()  # We'll add a shared legend later

# --- Shared legend ---
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=2,
    frameon=False
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig("longitudinal_tree_correspondence.svg", format="svg")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Categories corresponding to the 3 rows
row_categories = [
    "SNV_ONLY_OVERALL",
    "CNA_ONLY_OVERALL",
    "TREE_OVERALL"
]

metrics = ["Precision", "Recall", "F1 Score"]

fig, axes = plt.subplots(3, 3, figsize=(16, 16))

for row_idx, cat in enumerate(row_categories):
    df_cat = single_timepoint_results_df[single_timepoint_results_df["Category"] == cat]

    for col_idx, metric in enumerate(metrics):
        ax = axes[row_idx, col_idx]

        sns.boxplot(
            data=df_cat,
            x="Samples",
            y=metric,
            hue="Method",
            palette="deep",
            hue_order=["LoPhy", "COMPASS", "SCITE"],
            ax=ax
        )

        ax.set_title(f"{cat} — {metric}")
        ax.set_xlabel("Samples")
        ax.set_ylabel(metric)
        ax.legend_.remove()  # We'll add a shared legend later

# --- Shared legend ---
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=2,
    frameon=False
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig("single_timeponit_tree_correspondence.svg", format="svg")
plt.show()


In [ ]:
sample1_results_df[sample1_results_df.Recall < 0.4]

In [ ]:
results_df[results_df.Category == "SNV_SNV_ancestor_descendant"]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Categories corresponding to the 3 rows
row_categories = [
    "SNV_ONLY_OVERALL",
    "CNA_ONLY_OVERALL",
    "TREE_OVERALL"
]

metrics = ["Precision", "Recall", "F1 Score"]

fig, axes = plt.subplots(3, 3, figsize=(16, 16))

for row_idx, cat in enumerate(row_categories):
    df_cat = results_df[results_df["Category"] == cat]

    for col_idx, metric in enumerate(metrics):
        ax = axes[row_idx, col_idx]

        sns.boxplot(
            data=df_cat,
            x="Samples",
            y=metric,
            hue="Method",
            palette="deep",
            hue_order=["LoPhy", "COMPASS", "SCITE", "LACE"],
            ax=ax
        )

        ax.set_title(f"{cat} — {metric}")
        ax.set_xlabel("Samples")
        ax.set_ylabel(metric)
        ax.legend_.remove()  # We'll add a shared legend later

# --- Shared legend ---
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=2,
    frameon=False
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig("precision_recall_f1_snv_cna_tree.svg", format="svg")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Categories corresponding to the 3 rows
row_categories = [
    "SNV_SNV_ancestor_descendant",
    "SNV_SNV_coclustered",
    "SNV_SNV_separate_lineage"
]

metrics = ["Precision", "Recall", "F1 Score"]

fig, axes = plt.subplots(3, 3, figsize=(16, 16))

for row_idx, cat in enumerate(row_categories):
    df_cat = longitudinal_sim_results[longitudinal_sim_results["Category"] == cat]

    for col_idx, metric in enumerate(metrics):
        ax = axes[row_idx, col_idx]

        sns.boxplot(
            data=df_cat,
            x="Samples",
            y=metric,
            hue="Method",
            palette="deep",
            hue_order=["LoPhy", "COMPASS", "SCITE", "LACE"],
            ax=ax
        )

        ax.set_title(f"{cat} — {metric}")
        ax.set_xlabel("Samples")
        ax.set_ylabel(metric)
        ax.legend_.remove()  # We'll add a shared legend later

# --- Shared legend ---
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=2,
    frameon=False
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig("SNV_relationship_pr.svg", format="svg")
plt.show()


In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=results_f1, x="Samples", y="F1 Score", hue="Method", palette="deep", ax=ax, hue_order=["LoPhy", "COMPASS"])
ax.set_title("Tree Reconstruction F1 Score")
ax.set_ylabel("Tree Reconstruction F1 Score")
ax.set_xlabel("Samples")

In [ ]:
num_samples = 5
sim_path = os.path.join(path, f"sims_{num_samples}samples")
for sim_folder in os.listdir(sim_path):
    sim_directory = Path(os.path.join(sim_path, sim_folder))
    if sim_directory.is_dir():
        h5ad_fn = os.path.join(sim_directory, "adata.h5ad")
        adata = io.read_h5ad(h5ad_fn)
        T_true = load_tree(adata, "mutation_tree")
        _, T_LoPhy = op.io.load_dot(os.path.join(sim_directory, "LoPhy", "out_ml0.gv"), _type="cell_tree")
        samples = adata.var.SAMPLE.unique()
        for s in sorted(samples):
            print(f"Timepoint {s}: {list(adata.var.loc[adata.var.SAMPLE==s, "NAME"])}")
        print(results_df.loc[(results_df.Dataset == sim_folder) & results_df.Method.isin(["COMPASS","LoPhy"]) & results_df.Category.isin(["SNV_SNV_ancestor_descendant", "SNV_SNV_coclustered", "SNV_SNV_separate_lineage"])])
        op.pl.show_tree(T_LoPhy)
        op.pl.show_tree(T_true)

In [ ]:
sims5_samples_all_results = []
path = os.path.join(os.getcwd())
for num_samples in range(5,6):
    print(num_samples)
    sim_path = os.path.join(path, f"sims_{num_samples}samples")
    for sim_folder in os.listdir(sim_path):
        sim_directory = Path(os.path.join(sim_path, sim_folder))
        if sim_directory.is_dir():
            h5ad_fn = os.path.join(sim_directory, "adata.h5ad")
            print(h5ad_fn)
            adata = io.read_h5ad(h5ad_fn)
            T_true = load_tree(adata, "mutation_tree")
            _, T_LoPhy = op.io.load_dot(os.path.join(sim_directory, "LoPhy", "out_ml0.gv"), _type="cell_tree")

            results_gt = resolve_mutation_relationships(T_true)
            results_LoPhy = resolve_mutation_relationships(T_LoPhy, "LoPhy")

            scores_LoPhy = evaluate_relationships(results_gt, results_LoPhy)
                
            sims5_samples_all_results.append(scores_to_df(scores_LoPhy, sim_folder, num_samples, "LoPhy"))
sims_5samples_results_df = pd.concat(sims5_samples_all_results, ignore_index=True)


In [ ]:
sims_5samples_results_df

In [ ]:
num_samples = 5
sim_path = os.path.join(os.getcwd(), f"sims_{num_samples}samples")
for sim_folder in os.listdir(sim_path):
    sim_directory = Path(os.path.join(sim_path, sim_folder))
    if sim_directory.is_dir():
        h5ad_fn = os.path.join(sim_directory, "adata.h5ad")
        adata = io.read_h5ad(h5ad_fn)
        T_true = load_tree(adata, "mutation_tree")
        _, T_LoPhy = op.io.load_dot(os.path.join(sim_directory, "LoPhy", "out_ml0.gv"), _type="cell_tree")
        samples = adata.var.SAMPLE.unique()
        for s in sorted(samples):
            print(f"Timepoint {s}: {list(adata.var.loc[adata.var.SAMPLE==s, "NAME"])}")
        print(sims_5samples_results_df.loc[(sims_5samples_results_df.Dataset == sim_folder) & sims_5samples_results_df.Method.isin(["COMPASS","LoPhy"]) & sims_5samples_results_df.Category.isin(["SNV_SNV_ancestor_descendant", "SNV_SNV_coclustered", "SNV_SNV_separate_lineage"])])
        op.pl.show_tree(T_LoPhy)
        op.pl.show_tree(T_true)
        
#                                Category  Precision    Recall  F1 Score  
# 144  SNV_SNV_ancestor_descendant   0.963768  0.930070  0.946619  
# 148          SNV_SNV_coclustered   0.666667  0.952381  0.784314  
# 152     SNV_SNV_separate_lineage   1.000000  0.846154  0.916667  
# 162  SNV_SNV_ancestor_descendant   0.942446  0.916084  0.929078  
# 166          SNV_SNV_coclustered   0.800000  0.761905  0.780488  
# 170     SNV_SNV_separate_lineage   0.709677  0.846154  0.771930  


In [ ]:
results_df.loc[results_df.Dataset == ) & results_df.Method == "LoPhy", ]["SNV_SNV_ancestor_descendant", "SNV_SNV_coclusered", "SNV_SNV_separate_lineage"]]

In [ ]:
all_results = []
path = os.path.join(os.getcwd(), "simulations")
for num_samples in range(1,2):
    print(num_samples)
    sim_path = os.path.join(path, f"sims_{num_samples}samples")
    for sim_folder in os.listdir(sim_path):
        sim_directory = Path(os.path.join(sim_path, sim_folder))
        if sim_directory.is_dir():
            h5ad_fn = os.path.join(sim_directory, "adata.h5ad")
            adata = io.read_h5ad(h5ad_fn)
            T_true = load_tree(adata, "mutation_tree")
            _, T_LoPhy = op.io.load_dot(os.path.join(sim_directory, "LoPhy", "out_ml0.gv"), _type="cell_tree")
       
            results_gt = resolve_mutation_relationships(T_true)
            results_LoPhy = resolve_mutation_relationships(T_LoPhy, "LoPhy")

            scores_LoPhy = evaluate_relationships(results_gt, results_LoPhy)
                
            all_results.append(scores_to_df(scores_LoPhy, sim_folder, num_samples, "LoPhy"))
sample1_results_df = pd.concat(all_results, ignore_index=True)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Categories corresponding to the 3 rows
row_categories = [
    "SNV_ONLY_OVERALL",
    "CNA_ONLY_OVERALL",
    "TREE_OVERALL"
]

metrics = ["Precision", "Recall", "F1 Score"]

fig, axes = plt.subplots(3, 3, figsize=(16, 16))

for row_idx, cat in enumerate(row_categories):
    df_cat = sample1_results_df[sample1_results_df["Category"] == cat]

    for col_idx, metric in enumerate(metrics):
        ax = axes[row_idx, col_idx]

        sns.boxplot(
            data=df_cat,
            x="Samples",
            y=metric,
            hue="Method",
            palette="deep",
            hue_order=["LoPhy"],
            ax=ax
        )

        ax.set_title(f"{cat} — {metric}")
        ax.set_xlabel("Samples")
        ax.set_ylabel(metric)
        ax.legend_.remove()  # We'll add a shared legend later

# --- Shared legend ---
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=2,
    frameon=False
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
samples = adata.var.SAMPLE.unique()
for s in sorted(samples):
    print(f"Timepoint {s}: {list(adata.var.loc[adata.var.SAMPLE==s, "NAME"])}")
          

In [ ]:
op.pl.show_tree(T_true)

In [ ]:
op.pl.show_tree(T_COMPASS)

In [ ]:
cn_tree_parents, cn_tree_CNAs, cn_tree_dot_string = infer_copy_number_tree(np.array(parents), CNAs)

In [ ]:
cn_tree_parents

In [ ]:
cn_tree_parents

In [ ]:
cn_tree_CNAs

In [ ]:
parents

In [ ]:
CNAs

In [ ]:
from sklearn.metrics import adjusted_rand_score
ARI_results = []
path = os.path.join(os.getcwd(), "simulations")
for num_samples in range(2,6):
    print(num_samples)
    sim_path = os.path.join(path, f"sims_{num_samples}samples")
    for sim_folder in os.listdir(sim_path):
        sim_directory = Path(os.path.join(sim_path, sim_folder))
        print(sim_folder)
        if sim_directory.is_dir():
            h5ad_fn = os.path.join(sim_directory, "adata.h5ad")
            adata = io.read_h5ad(h5ad_fn)
            T_true = load_tree(adata, "mutation_tree")
            true_cell_assignments = T_true.graph["cell_assignments"]

            _, T_LoPhy = op.io.load_dot(os.path.join(sim_directory, "LoPhy", "out_ml0.gv"), _type="cell_tree")
            LoPhy_cluster_assignments = np.array(T_LoPhy.graph["cell_assignments"], dtype=int)
            ARI_results.append([sim_folder, "LoPhy", num_samples, adjusted_rand_score(true_cell_assignments, LoPhy_cluster_assignments)])
            
            T_COMPASS = nx.nx_pydot.read_dot(os.path.join(sim_directory, "COMPASS", "out_tree.gv"))
            COMPASS_cell_assignments_df = pd.read_csv(os.path.join(sim_directory, "COMPASS", "out_cellAssignments.tsv"), header=0, index_col=0, sep="\t")
            COMPASS_cell_assignments = COMPASS_cell_assignments_df["node"].values
            ARI_results.append([sim_folder, "COMPASS", num_samples, adjusted_rand_score(true_cell_assignments, COMPASS_cell_assignments)])
            
            cells = adata.obs.index.to_series().values
            mapping = {'s%d' % i:cells[i] for i in range(len(cells))}
            T_SCITE_cell, T_SCITE = op.io.load_dot(os.path.join(sim_directory, "SCITE", "output_ml0.gv"), 
                                       mutations = list(adata.var.index), 
                                       cells = list(adata.obs.index), 
                                       mapping=mapping,
                                       _type="cell_tree")
            (_, SCITE_ccm) = op.ul.resolve_genotypes(T_SCITE_cell, adata.X)

            SCITE_ccm["cluster"] = pd.factorize(
                [tuple(row) for row in SCITE_ccm.to_numpy()]
            )[0]

            SCITE_cluster_assignments = SCITE_ccm["cluster"].values
            ARI_results.append([sim_folder, "SCITE", num_samples, adjusted_rand_score(true_cell_assignments, SCITE_cluster_assignments)])

            mapping = {'s%d' % i:cells[i] for i in range(len(cells))}
            T_LACE_cell, T_LACE = op.io.load_dot(os.path.join(sim_directory, "LACE", "tree.gv"),
                                       mutations = list(adata.var.index), 
                                       cells = list(adata.obs.index), 
                                       mapping=mapping,
                                       _type="cell_tree")
            (_, LACE_ccm) = op.ul.resolve_genotypes(T_LACE_cell, adata.X)
            
            LACE_ccm["cluster"] = pd.factorize(
                [tuple(row) for row in LACE_ccm.to_numpy()]
            )[0]

            LACE_cluster_assignments = LACE_ccm["cluster"].values
            ARI_results.append([sim_folder, "LACE", num_samples, adjusted_rand_score(true_cell_assignments, LACE_cluster_assignments)])


In [ ]:
ARI_results_df = pd.DataFrame(ARI_results, columns=["Dataset", "Model", "Samples", "ARI"])

In [ ]:
ARI_results_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots()

sns.boxplot(
    data=ARI_results_df,
    x="Samples",
    y="ARI",
    hue="Model",
    palette="deep",
    hue_order=["LoPhy", "COMPASS", "SCITE", "LACE"],
    ax=ax
)

# Remove the axes legend and create a shared figure legend
handles, labels = ax.get_legend_handles_labels()
ax.get_legend().remove()

fig.legend(
    handles,
    labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.05),
    ncol=len(labels),
    frameon=False
)

plt.tight_layout(rect=[0, 0, 1, 0.98])  # Leave space for legend
fig.savefig("ARI_metric.svg", format="svg")
plt.show()

In [ ]:
datasets = ARI_results_df["Dataset"].unique()

In [ ]:
compass_better_vals = []
scite_better_vals = []
for d in datasets:
    COMPASS_ari_val = ARI_results_df.loc[(ARI_results_df["Dataset"] == d) & (ARI_results_df["Model"] == "COMPASS"), "ARI"].values[0]
    LoPhy_ari_val = ARI_results_df.loc[(ARI_results_df["Dataset"] == d) & (ARI_results_df["Model"] == "LoPhy"), "ARI"].values[0]
    SCITE_ari_val = ARI_results_df.loc[(ARI_results_df["Dataset"] == d) & (ARI_results_df["Model"] == "SCITE"), "ARI"].values[0]

    print(d)
    print(f"COMPASS ARI: {COMPASS_ari_val} \n LoPhy ARI: {LoPhy_ari_val} \n SCITE ARI: {SCITE_ari_val}") 
    if COMPASS_ari_val > LoPhy_ari_val:
        diff = COMPASS_ari_val - LoPhy_ari_val
        print(f"COMPASS is better by {diff}")
        compass_better_vals.append(diff)
    if SCITE_ari_val > LoPhy_ari_val:
        diff = SCITE_ari_val - LoPhy_ari_val
        print(f"SCITE is better by {diff}")
        scite_better_vals.append(diff)
    print()

In [ ]:
np.mean(compass_better_vals)

In [ ]:
np.mean(scite_better_vals)